# $\phi$ \ $\chi$ spectra
Comparing observed spectra of acoustic phase, $\phi$ and log-amplitude, $\chi$ with modelled spectra given Garrett-Munk internal waves spectra (calculated with WFT).

In [ ]:
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from dotenv import find_dotenv, load_dotenv
from scipy import signal, optimize

from n23_fluctuations_ragland import tools
import warnings
import colorsys

_ = load_dotenv(find_dotenv())

%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2

### Poster presentation settings

In [ ]:
import matplotlib.font_manager as fm
from nescafe23 import get_data_dir

font_path = str(get_data_dir() / "fonts" / "RedHatDisplay-Medium.ttf")
fm.fontManager.addfont(font_path)

plt.rcParams['font.family'] = 'Red Hat Display'

In [ ]:
# Define your colors
whoi_colors = ['#00B7BD', '#EE5340']

In [ ]:
def lighten_color(hex_color, saturation_scale=0.5, lightness_scale=1.5):
    """
    Lighten a hex color similar to seaborn's tab20 palette
    
    Args:
        hex_color: Hex color string (e.g., '#1f77b4')
        saturation_scale: Multiply saturation by this (default 0.5)
        lightness_scale: Multiply lightness by this (default 1.5)
    """
    # Remove '#' if present
    hex_color = hex_color.lstrip('#')
    
    # Convert hex to RGB (0-1 range)
    r, g, b = tuple(int(hex_color[i:i+2], 16) / 255.0 for i in (0, 2, 4))
    
    # Convert RGB to HLS
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    
    # Adjust saturation and lightness
    s_new = min(1.0, s * saturation_scale)
    l_new = min(1.0, l * lightness_scale)
    
    # Convert back to RGB
    r_new, g_new, b_new = colorsys.hls_to_rgb(h, l_new, s_new)
    
    # Convert to hex
    return '#{:02x}{:02x}{:02x}'.format(
        int(r_new * 255),
        int(g_new * 255),
        int(b_new * 255)
    )

In [ ]:
colors = [
    whoi_colors[0],
    lighten_color(whoi_colors[0], lightness_scale=2),
    whoi_colors[1],
    lighten_color(whoi_colors[1], lightness_scale=1.4, saturation_scale=1)
]

In [ ]:
colors

In [ ]:
plt.plot(np.arange(0,10), lw=2, c=colors[0])
plt.plot(np.arange(2,12), lw=2, c=colors[1])
plt.plot(np.arange(4,14), lw=2, c=colors[2])
plt.plot(np.arange(6,16), lw=2, c=colors[3])

## open observations and fit parameters

In [ ]:
m_data, F_phi_data, F_chi_data, F_phi_lm, F_phi_ls, F_chi_lm, F_chi_ls = tools.estimate_F_phi_chi()

## Compute internal wave spectra (WFT)

In [ ]:
fn = f"{os.environ['NESCAFE23_DATADIR']}dataproducts/phi_chi_LS_fit.pkl"

with open(fn, 'rb') as f:
    results = pickle.load(f)

## Plot Spectra

In [ ]:
m = 2 * np.pi * np.logspace(-5, 0, 500)[None, :]  # [rad / m]

In [ ]:
plt.rcParams.update({"font.size": 8})
plt.rcParams["text.usetex"] = False
sns.set_palette("tab20")

fig, axes = plt.subplots(2, 2, figsize=(6.67, 5))
axes = axes.flatten()

for k, ray_id in enumerate(["+7", "-7", "+6", "-6"]):
    
    F_phi_model, F_chi_model, F_phi_model_nonoise, F_chi_model_nonoise = tools.phi_chi_spectrum_model(m, results[ray_id].x[0], results[ray_id].x[1], results[ray_id].x[2], ray_id)

    # extend noise floor to the right
    nan_mask = np.isnan(F_phi_model)
    F_phi_model = np.hstack((F_phi_model[~nan_mask], F_phi_model[~nan_mask][-1]))
    F_chi_model = np.hstack((F_chi_model[~nan_mask], F_chi_model[~nan_mask][-1]))
    m_extend = np.concatenate((m[:,~nan_mask], np.array([[100]])), axis=1)
    
    plt.sca(axes[k])

    # WFT GMIW F_phi
    plt.plot(m[0] / (2 * np.pi), F_phi_model_nonoise, c=colors[0], lw=1.5)
    plt.plot(m_extend[0] / (2 * np.pi), F_phi_model, c=colors[0], lw=0.75, linestyle='dashed')
    
    # F_phi observed
    plt.plot(m_data, F_phi_lm[ray_id], c=colors[1], marker="o", markersize=3)
    plt.fill_between(
        m_data,
        F_phi_lm[ray_id] * F_phi_ls[ray_id],
        F_phi_lm[ray_id] / F_phi_ls[ray_id],
        alpha=0.3,
        color=colors[1],
        lw=0,
    )

    # WFT GMIW F_chi
    plt.plot(m[0] / (2 * np.pi), F_chi_model_nonoise, c=colors[2], lw=1.5)
    plt.plot(m_extend[0] / (2 * np.pi), F_chi_model, c=colors[2], lw=0.75, linestyle='dashed')
    
    # F_chi observed
    plt.plot(m_data, F_chi_lm[ray_id], c=colors[3], marker="o", markersize=3)
    plt.fill_between(
        m_data,
        F_chi_lm[ray_id] * F_chi_ls[ray_id],
        F_chi_lm[ray_id] / F_chi_ls[ray_id],
        alpha=0.3,
        color=colors[3],
        lw=0,
    )

    # Observational Noise Floor
    #plt.hlines([F_phi_obs_noise[ray_id], F_chi_obs_noise[ray_id]], 1e-2, 1e-0, colors=['C0', 'C2'])
    #print(F_phi_obs_noise[ray_id], F_chi_obs_noise[ray_id])
    
    plt.ylim([2e-4, 3e4])
    plt.xlim([2e-4, 4e-1])

    plt.xlabel("vertical wave number [1/m]")
    plt.ylabel("$F_{\\phi, \\chi}$ [$\\mathrm{rad} \\ m$]")

    plt.yscale("log")
    plt.xscale("log")

    plt.grid()

    plt.text(
        0.7,
        0.9,
        f"Ray ID: {ray_id}",
        transform=axes[k].transAxes,
        fontsize=9,
        bbox=dict(facecolor="white", edgecolor="k", pad=3, linewidth=0.5),
    )
    
    plt.text(
        0.04,
        0.07,
        f"{results[ray_id].x[0]:0.1f} $\\times$ GM Energy\n$F_{{\\phi}}$ noise level = {results[ray_id].x[1]:0.2f} [rad m]\n$F_{{\\chi}}$ noise level = {results[ray_id].x[2]:0.2f} [rad m]",
        transform=axes[k].transAxes,
        fontsize=8,
        bbox=dict(facecolor="white", edgecolor="k", pad=3, linewidth=0.5)
    )

plt.tight_layout()

fig.savefig('../../figures/presentation_figures/F_phi_chi_ducted.pdf', bbox_inches='tight')